# Futsal Match Analysis — 10-minute clip on Colab
Runtime → Change runtime type → **T4 GPU** (or better). Then run cells top to bottom.

In [ ]:
# 1. GPU check + clone repo
!nvidia-smi -L
!git clone https://github.com/azamat-byte/futsal.git /content/futsal
%cd /content/futsal

In [ ]:
# 2. Install (takes ~3-5 min)
!pip install -q -e ".[phase1,rfdetr,embeddings,colab]"
!apt-get -qq install -y ffmpeg > /dev/null

In [ ]:
# 3. API key (Roboflow models for detection / court keypoints / pose)
# Add ROBOFLOW_API_KEY in Colab's left sidebar -> Secrets (key icon)
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')

In [ ]:
# 4. Get the video: mount Drive (recommended for big files) OR upload
from google.colab import drive
drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/futsal/match.mp4'   # <-- change to your file
# Alternative: from google.colab import files; SRC = list(files.upload())[0]

In [ ]:
# 5. Cut a 10-minute clip and normalise to 25 fps / 1080p (keeps memory & timings predictable)
START = '00:00:00'     # where the 10 minutes begin in the source
!ffmpeg -y -loglevel error -ss {START} -i "{SRC}" -t 600 -vf "fps=25,scale=-2:1080" -c:v libx264 -preset fast -crf 20 -c:a aac clip.mp4
import cv2
cap = cv2.VideoCapture('clip.mp4')
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); FPS = cap.get(cv2.CAP_PROP_FPS); cap.release()
print(N, 'frames @', FPS, 'fps')   # expect ~15000

In [ ]:
# 6. Phase 1 — detection + ByteTrack + team clustering (longest step: ~25-45 min on T4)
RUNS = '/content/drive/MyDrive/futsal/runs'   # saved to Drive so a disconnect doesn't lose work
!mkdir -p {RUNS}
!python -m futsal.phase1 --video clip.mp4 --output {RUNS}/p1 --config configs/phase1.json --max-frames {N}

In [ ]:
# 7. Phase 2 — automatic court calibration (keypoints every 30 frames) + 2D pitch projection
!python -m futsal.phase2 auto --video clip.mp4 --output {RUNS}/calibration.json --max-frames {N} --stride 30
!python -m futsal.phase2 render --phase1-run {RUNS}/p1 --calibration {RUNS}/calibration.json --output {RUNS}/p2

In [ ]:
# 8. Phase 3 (speed / distance / Voronoi) and Phase 4 (ball, possession, pose)
!python -m futsal.phase3 --phase2-run {RUNS}/p2 --output {RUNS}/p3
!python -m futsal.phase4 --phase2-run {RUNS}/p2 --phase3-run {RUNS}/p3 --output {RUNS}/p4 --config configs/phase1.json --device 0

In [ ]:
# 9. Phase 6 — jersey-number OCR (SmolVLM) + Re-ID stitching -> writes jersey_number into Phase 4 tracks
import pandas as pd, numpy as np, cv2
from futsal.player_id_ocr import VLMJerseyOCR
from futsal.reid_boundary import PlayerIdentityManager, InPlayBoundaryFilter

ocr = VLMJerseyOCR(model_id='HuggingFaceTB/SmolVLM-256M-Instruct', device='cuda')
mgr = PlayerIdentityManager(ocr=ocr)
bnd = InPlayBoundaryFilter()
df = pd.read_csv(f'{RUNS}/p4/possession_tracks.csv')
by_frame = {int(f): g.to_dict('records') for f, g in df.groupby('frame_index')}

out, cap = [], cv2.VideoCapture('clip.mp4')
for i in range(N):
    ok, frame = cap.read()
    if not ok: break
    rows = by_frame.get(i)
    if not rows: continue
    rows = bnd.filter_frame_rows(rows)
    out += mgr.update_frame(frame, rows, i)
    if i % 1500 == 0: print(f'{i}/{N}  profiles={len(mgr.profiles)}')
cap.release()
pd.DataFrame(out).to_csv(f'{RUNS}/p4/possession_tracks.csv', index=False)
for (team, num), p in sorted(mgr.profiles.items()):
    print(team, f'#{num}', p.player_name, 'raw track ids:', p.historical_tracker_ids)

In [ ]:
# 10. Phase 5 — broadcast video + SQLite telemetry (runs/p5/telemetry.db) + shot events
!python -m futsal.phase5 --phase2-run {RUNS}/p2 --phase3-run {RUNS}/p3 --phase4-run {RUNS}/p4 --output {RUNS}/p5 --video clip.mp4

In [ ]:
# 11. Match stats grouped by jersey number (SQLite)
import sqlite3
from futsal.telemetry_sqlite import PLAYER_SUMMARY_BY_JERSEY_SQL
con = sqlite3.connect(f'{RUNS}/p5/telemetry.db')
display(pd.read_sql(PLAYER_SUMMARY_BY_JERSEY_SQL, con))
display(pd.read_sql("SELECT * FROM match_events WHERE event_type='Shot Made'", con))

In [ ]:
# 12. Download the video (or just open it from Drive)
from google.colab import files
files.download(f'{RUNS}/p5/broadcast_tactical.mp4')